# OpenAI Parameters

# Overview
When making requests to OpenAI models, several parameters can be used to control the behavior and output of the model. \
Understanding these parameters helps in fine-tuning the responses to meet specific requirements, whether for generating text, answering questions, or any other use case.

For more detailed examples, refer to the official documentation [Azure OpenAI Service](https://learn.microsoft.com/en-us/azure/ai-services/openai/reference)


In [4]:
import re
import requests
import sys
import os
from openai import AzureOpenAI
import tiktoken
from dotenv import load_dotenv
load_dotenv()

client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"), 
  api_key=os.getenv("AZURE_OPENAI_KEY"),  
  #api_version="2024-02-15-preview"
  #api_version="2024-12-01-preview",
  api_version="2024-10-21",
)

CHAT_COMPLETIONS_MODEL = os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')
SEED = 123

# Parameter: max_tokens
**Description**: The maximum number of tokens to generate in the completion. \
**Default**: 16 \
**Example**: max_tokens=50

The token count of your prompt plus max_tokens can't exceed the model's context length. \
Most models have a context length of 2048 tokens (except for the newest models, which support 4096). Please refer to documentation.

In [5]:
def call_openai_with_max_tokens(max_tokens):
    response = client.chat.completions.create(
          model=CHAT_COMPLETIONS_MODEL,
          messages = [{"role":"system", "content":"あなたは優れたペットの専門家です。"},
                    {"role":"user","content": "最高のペットは"}],
                    max_tokens=max_tokens
    )
    return response.choices[0].message.content

# Generate with different presence_penalty values
penalties = [16, 32, 60, 100]
for penalty in penalties:
    print(f"Max Tokens: {penalty}\n")
    print(call_openai_with_max_tokens(penalty))
    print("\n" + "-"*80 + "\n")

Max Tokens: 16

「最高のペット」と一言で言っても、それは飼

--------------------------------------------------------------------------------

Max Tokens: 32

「最高のペット」は人によって異なります。それは個々のライフスタイル、住環境、性格、特定の

--------------------------------------------------------------------------------

Max Tokens: 60

ペットとしての「最高」を決めるのは大変難しいです！なぜなら、どのペットが最も適しているかは、飼い主のライフスタイル、居住環境、個人的な好み、そしてペットの

--------------------------------------------------------------------------------

Max Tokens: 100

「最高のペット」とは、その人のライフスタイル、性格、住環境、そしてペットに対する期待に応じて異なります。つまり、万人にとっての「最高のペット」は存在せず、個々人にとって最適のペットがあるということです。以下では、各状況に合わせて考えるべきポイントやおすすめのペットを紹介します。

---

### **1. ラ

--------------------------------------------------------------------------------



# Parameter: temperature

**Description**: Controls the randomness of the output. Lower values make the output more deterministic, while higher values increase randomness. \
**Value Range**: 0 to 1 \
**Default Value**: 1 \
**Example**: temperature=0.7

Higher values means the model will take more risks. \
Try 0.9 for more creative applications, and 0 (argmax sampling) for ones with a well-defined answer.

---
**NOTE**: We generally recommend altering this or top_p but not both.


In [6]:
def call_openai(num_times, prompt, temperature=0.75, use_seed=False):
    for i in range(num_times):
        if use_seed:
            response = client.chat.completions.create(
                model=CHAT_COMPLETIONS_MODEL,
                messages = [{"role":"system", "content":"あなたは優れたペットの専門家です。"},
                            {"role":"user","content": prompt}],
                    max_tokens=32,
                    seed=SEED,
                    temperature = temperature
            )
        else:
            response = client.chat.completions.create(
                model=CHAT_COMPLETIONS_MODEL,
                messages = [{"role":"system", "content":"あなたは優れたペットの専門家です。"},
                            {"role":"user","content": prompt}],
                    max_tokens=32,
                    temperature = temperature
            )
        print(response.choices[0].message.content)

In [7]:
# Without seed and temperature, the response is different each time
call_openai(10, '最高のペットは')

「最高のペット」は個々のライフスタイル、好み、家庭環境によって異なります。人によって最高のペ
最高のペットは、その人のライフスタイルや好みによって異なります！どんなペットが「最高」かを考
ペットとして何を「最高」と感じるかは、人それぞれのライフスタイル、興味、ニーズ、そして住環境
ペットとして何を「最高」と考えるかは、人それぞれの生活スタイル、好み、ニーズによって異なります。
「最高のペット」は、人それぞれのライフスタイル、住環境、好み、そしてペットに対してどのくら
「最高のペット」というのは、人それぞれのライフスタイル、好み、生活環境によって異なります。どんな
ペットの「最高」を決めることは、人それぞれのライフスタイル、好み、ニーズによって異なります。
「最高のペット」は人それぞれのライフスタイル、性格、住環境、予算、そして好みによって異
ペットとして「最高」と言える種類は、飼い主のライフスタイル、性格、住環境、お世話にか
「最高のペット」は、個人のライフスタイル、好み、生活環境、そしてペットに費やせる時間や


In [8]:
# Now using a seed and 0 temperature, the response is the much more consisitent
call_openai(10, '最高のペットは', temperature = 0, use_seed=True)

「最高のペット」は人それぞれのライフスタイル、性格、住環境、そしてペットに求めるものによ
「最高のペット」とは、個人のライフスタイル、好み、住環境、そしてペットに費やせる時間
「最高のペット」は、飼い主のライフスタイル、性格、住環境、そしてペットに対する期待によ
「最高のペット」とは、個々のライフスタイル、好み、住環境、そしてペットに費やせる時間
「最高のペット」とは、個々のライフスタイル、好み、住環境、そしてペットに費やせる時間
「最高のペット」は人それぞれのライフスタイル、性格、住環境、そしてペットに求めるものによ
「最高のペット」とは、個人のライフスタイル、好み、住環境、そしてペットに費やせる時間
「最高のペット」とは、個人のライフスタイル、好み、住環境、そしてペットに費やせる時間
「最高のペット」とは、個々のライフスタイル、好み、住環境、そしてペットに費やせる時間
「最高のペット」は、飼い主のライフスタイル、性格、住環境、そしてペットに対する期待によ


# Parameter: n
**Description**: Specifies the number of completions to generate for each prompt. \
**Default Value**: 1 \
**Example**: n = 3 

---
**Note**: Because this parameter generates many completions, it can quickly consume your token quota. Use carefully and ensure that you have reasonable settings for max_tokens and stop.

In [9]:
response = client.chat.completions.create(
        model=CHAT_COMPLETIONS_MODEL,
        messages = [{"role":"system", "content":"あなたは優れたペットの専門家です。"},
                {"role":"user","content": "最高のペットは"}],
                max_tokens=60,
                n=2
        )

for index, c in enumerate(response.choices):
    print(index, c.message.content)

0 「最高のペット」は人それぞれのライフスタイル、好み、住環境、そしてペットに対して割ける時間やエネルギーに大きく依存します。同じ動物でも個性が異なるため、どのペ
1 「最高のペット」とは、個人のライフスタイル、嗜好、住環境、時間的余裕や予算など、さまざまな要因によって異なります。それぞれの動物が持つ特性や個性を


# Parameter: presence_penalty
**Description**: Penalizes new tokens based on whether they appear in the text so far, encouraging the model to use new tokens. \
**Value Range**: -2.0 to 2.0 \
**Default Value**: 0 \
**Example**: presence_penalty=0.5

In [10]:
def call_openai_with_presence_penalty(presence_penalty):
    response = client.chat.completions.create(
          model=CHAT_COMPLETIONS_MODEL,
            messages = [{"role":"system", "content":"あなたは優れたペットの専門家です。"},
                {"role":"user","content": "最高のペットは"}],
                    max_tokens=60,
                    presence_penalty=presence_penalty
    )
    return response.choices[0].message.content

# Generate with different presence_penalty values
penalties = [0, 0.5, 1.0, 1.5, 2.0]
for penalty in penalties:
    print(f"Presence Penalty: {penalty}\n")
    print(call_openai_with_presence_penalty(penalty))
    print("\n" + "-"*80 + "\n")

Presence Penalty: 0

「最高のペット」は、飼い主のライフスタイル、性格、住環境、お世話にどれだけ時間を割けるかといった多くの要因によって異なります。それぞれの人にとって「最高の

--------------------------------------------------------------------------------

Presence Penalty: 0.5

「最高のペット」というのは、実際のところ飼い主さんのライフスタイルや好みによって大きく異なります。人それぞれの生活環境や飼育可能な時間、まめに付き合えるかどうか

--------------------------------------------------------------------------------

Presence Penalty: 1.0

最高のペットは、あなたの生活スタイル、個人的な好み、そしてペットに対するコミットメントの程度によって異なります。どの動物が「最高」かは一概には決められないため、いくつかの

--------------------------------------------------------------------------------

Presence Penalty: 1.5

最高のペットといえるかどうかは、あなたのライフスタイルや好み、住環境によるところが大きいです。一人ひとりにとって「最高のペット」は異なりますので、以下にいくつかの選

--------------------------------------------------------------------------------

Presence Penalty: 2.0

「最高のペット」という概念は、個々のライフスタイルや好みによって異なるため、一概には言えません。あなたの状況、性格、そしてペットにどれだけの時間とリソースを割くことができる

--------------------------------------------------------------------------------



# Parameter: frequency_penalty
**Description**: Penalizes new tokens based on their existing frequency in the text so far, reducing the likelihood of repeating the same line verbatim. \
**Value Range**: -2.0 to 2.0 \
**Default Value**: 0 \
**Example**: frequency_penalty=0.5

#### Use cases to explore
1. **Compare Responses** \
Generate multiple completions to compare and choose the best response for your use case.

2. **Increase Diversity** \
Use multiple completions to get a variety of responses, which is useful in creative applications.

3. **Enhance Robustness** \
Generate multiple responses to ensure consistency and accuracy across different completions.

#### Best Practices
1. **Optimize Prompt Length** \
Keep your prompts concise but informative to ensure the model has enough context.

2. **Adjust Temperature and Top_p** \
Use these parameters to balance between deterministic and creative responses.

3. **Monitor Token Usage** \
Be mindful of the max_tokens parameter to manage costs and response length.

4. **Use Stopping Sequences** \
Define stopping sequences to control where the model should stop generating text, ensuring the output is within the desired context.

5. **Generate Multiple Completions** \
Use the n parameter to generate multiple completions and select the best one for your needs.